In [6]:
import re
import glob
import os
import csv

RESULT_DIR = '/home/yichi/research/w4_17/w4-17_all/result/molout'
OUT_DIR    = '/home/yichi/research/w4_17/w4-17_all/summary/vtzfp'

DIRS = {
    'closed': os.path.join(RESULT_DIR, 'closed_shell', 'tz'),
    'open':   os.path.join(RESULT_DIR, 'open_shell',   'tz'),
}

In [7]:
ENERGY_RE = re.compile(r'Final AFQMC/pt2CCSD energy:\s*([-\d.]+)\s*\u00b1\s*([\d.]+)')

def extract_afqmc(fpath):
    """Return (energy, error) from an AFQMC output file, or (None, None)."""
    with open(fpath) as f:
        for line in f:
            m = ENERGY_RE.search(line)
            if m:
                return float(m.group(1)), float(m.group(2))
    return None, None

In [8]:
rows = []
missing = []

for shell, d in DIRS.items():
    for fpath in sorted(glob.glob(os.path.join(d, '*_afqmc_vtzfp.out'))):
        mol = os.path.basename(fpath).replace('_afqmc_vtzfp.out', '')
        energy, error = extract_afqmc(fpath)
        if energy is None:
            missing.append((mol, shell, fpath))
        rows.append({'molecule': mol, 'shell': shell, 'E_AFQMC': energy, 'E_AFQMC_err': error})

print(f'Total molecules extracted : {len(rows)}')
print(f'Missing energy            : {len(missing)}')
if missing:
    for mol, shell, fp in missing:
        print(f'  {shell:6s}  {mol:30s}  {fp}')

Total molecules extracted : 200
Missing energy            : 0


In [9]:
# Write CSV
out_csv = os.path.join(OUT_DIR, 'afqmc_vtzfp.csv')
with open(out_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['molecule', 'shell', 'E_AFQMC', 'E_AFQMC_err'])
    writer.writeheader()
    writer.writerows(rows)

print(f'Saved: {out_csv}')

Saved: /home/yichi/research/w4_17/w4-17_all/summary/vtzfp/afqmc_vtzfp.csv


In [10]:
# Write fixed-width dat file
out_dat = os.path.join(OUT_DIR, 'afqmc_vtzfp.dat')
header  = f"{'Molecule':<30} {'Shell':<8} {'E_AFQMC':>16} {'E_AFQMC_err':>14}\n"
sep     = '-' * 72 + '\n'

with open(out_dat, 'w') as f:
    f.write(header)
    f.write(sep)
    for r in rows:
        e   = f"{r['E_AFQMC']:>16.6f}"   if r['E_AFQMC']     is not None else f"{'missing':>16}"
        err = f"{r['E_AFQMC_err']:>14.6f}" if r['E_AFQMC_err'] is not None else f"{'':>14}"
        f.write(f"{r['molecule']:<30} {r['shell']:<8} {e} {err}\n")

print(f'Saved: {out_dat}')

Saved: /home/yichi/research/w4_17/w4-17_all/summary/vtzfp/afqmc_vtzfp.dat


In [11]:
# Summary by shell type
closed = [r for r in rows if r['shell'] == 'closed' and r['E_AFQMC'] is not None]
open_  = [r for r in rows if r['shell'] == 'open'   and r['E_AFQMC'] is not None]

print(f'Closed-shell molecules : {len(closed)}')
print(f'Open-shell molecules   : {len(open_)}')
print()
print(f"{'Molecule':<30} {'Shell':<8} {'E_AFQMC':>16} {'± err':>12}")
print('-' * 72)
for r in rows:
    e   = f"{r['E_AFQMC']:>16.6f}"    if r['E_AFQMC']     is not None else f"{'missing':>16}"
    err = f"{r['E_AFQMC_err']:>12.6f}" if r['E_AFQMC_err'] is not None else f"{'':>12}"
    print(f"{r['molecule']:<30} {r['shell']:<8} {e} {err}")

Closed-shell molecules : 160
Open-shell molecules   : 40

Molecule                       Shell             E_AFQMC        ± err
------------------------------------------------------------------------
acetaldehyde                   closed        -153.581313     0.000236
acetic                         closed        -228.747734     0.000283
alcl3                          closed       -1621.420486     0.000172
alcl                           closed        -701.789294     0.000100
alf3                           closed        -541.444964     0.000271
alf                            closed        -341.799736     0.000240
alh3                           closed        -243.759338     0.000026
alh                            closed        -242.544439     0.000036
allene                         closed        -116.428977     0.000238
b2h6                           closed         -53.135669     0.000066
benzene                        closed        -231.799019     0.000334
beta-lactim                  